##### Single Ingest

In [38]:
from datetime import datetime

class SearchEngine:
    def __init__(self):
        self.schema = {}
        self.index = {}
        self.documents = {}
        self.next_doc_id = 0

    # -----------------------------
    # Determine field type
    # -----------------------------
    def determine_type(self, value):
        if isinstance(value, bool):
            return "bool"
        elif isinstance(value, int):
            return "int"
        elif isinstance(value, float):
            return "float"
        elif isinstance(value, str):
            try:
                datetime.strptime(value, "%Y-%m-%d")
                return "date"
            except ValueError:
                return "categorical"
        return "unknown"

    # -----------------------------
    # Build schema
    # -----------------------------
    def build_schema(self, doc):
        for key, value in doc.items():
            field_type = self.determine_type(value)
            self.schema[key] = {
                "type": field_type,
                "indexed": field_type == "categorical"
            }
    # -----------------------------
    # Index one document
    # -----------------------------
    def index_document(self, doc):
        # Build schema only once
        if not self.schema:
            self.build_schema(doc)
        doc_id = self.next_doc_id
        # Save original document
        self.documents[doc_id] = doc
        # Update bitmap index
        for field, value in doc.items():
            # Ignore fields that aren't indexed
            if not self.schema[field]["indexed"]:
                continue
            if field not in self.index:
                self.index[field] = {}
            if value not in self.index[field]:
                self.index[field][value] = set()
            self.index[field][value].add(doc_id)
        self.next_doc_id += 1

    # -----------------------------
    # Equality lookup
    # -----------------------------
    def lookup(self, field, value):
        if field not in self.index:
            return set()
        if value not in self.index[field]:
            return set()
        return self.index[field][value]

    def query_and(self, results):
        if not results:
            return set()
        answer = results[0].copy()
        for result in results[1:]:
            answer &= result
        return answer

    def query_or(self, results):
        if not results:
            return set()
        answer = results[0].copy()
        for result in results[1:]:
            answer |= result
        return answer

    def query_not(self, result):
        all_docs = set(self.documents.keys())
        return all_docs - result

    def query_in(self, field, values):
        results = []
        for value in values:
            results.append(self.lookup(field, value))
        return self.query_or(results)

    def query_not_in(self, field, values):
        result = self.query_in(field, values)
        return self.query_not(result)

    # batch ingest
    def index_documents(self, docs):
        for doc in docs:
            self.index_document(doc)

    #cardinality Analysis
    def get_cardinality(self):
        cardinality={}
        for field in self.index:
            cardinality[field]=len(self.index[field])
        return cardinality

    def analyse_cardinality(self):
        cardinality=self.get_cardinality()
        for field,count in cardinality.items():
            self.schema[field]["cardinality"] = count;

    def disable_high_cardinality(self,threshold):
        self.analyse_cardinality()
        for field in self.schema:
            if self.schema[field]["cardinality"]>threshold:
                self.schema[field]["indexed"]=False

In [39]:
doc1 = {
    "name": "Alex",
    "theme": "dark",
    "country": "USA"
}

doc2 = {
    "name": "John",
    "theme": "dark",
    "country": "India"
}

doc3 = {
    "name": "Alex",
    "theme": "light",
    "country": "India"
}


engine = SearchEngine()

engine.index_document(doc1)
engine.index_document(doc2)
engine.index_document(doc3)

In [41]:
doc4 = {
    "name": "Pragya",
    "theme": "light",
    "country": "India"
}
engine.index_document(doc4)

In [42]:
print(engine.get_cardinality())

{'name': 3, 'theme': 2, 'country': 2}


In [ ]:
engine.disable_high_cardinality(2)

In [44]:
doc5 = {
    "name": "Rachna",
    "theme": "light",
    "country": "India"
}
engine.index_document(doc5)

In [ ]:
#name not updated becoz name cardinality false
print(engine.get_cardinality())

{'name': 3, 'theme': 2, 'country': 2}


In [46]:
engine.lookup("name", "Alex")
# {0,2}

{0, 2}

In [47]:
engine.query_and([
    engine.lookup("name","Alex"),
    engine.lookup("theme","dark")
])
# {0}

{0}

In [34]:
engine.query_or([
    engine.lookup("name","Alex"),
    engine.lookup("country","India")
])

{0, 1, 2}

In [35]:
engine.query_not(
    engine.lookup("country","USA")
)
# {1,2}

{1, 2}

In [36]:
engine.query_in(
    "country",
    ["USA","India"]
)
# {0,1,2}

{0, 1, 2}

In [37]:
engine.query_not_in(
    "theme",
    ["dark"]
)
# {2}

{2}